In [73]:
from IPython.display import display, HTML

display(HTML("""
<style>

/* =========================
   전체 레이아웃
========================= */

div.container{
    width:85% !important;
}

div.cell.code_cell.rendered{
    width:100%;
}

div.input_prompt{
    padding:0;
}

div.prompt{
    min-width:70px;
}

div#toc-wrapper{
    padding-top:120px;
}

table.dataframe{
    font-size:12px;
}

/* =========================
   코드 입력창
========================= */

div.CodeMirror{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
    line-height:1.6;
}

/* =========================
   입력 셀
========================= */

div.input{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
}

/* =========================
   코드 출력
========================= */

div.output{
    font-family:"마루 부리OTF 중간" !important;
    font-size:12pt !important;
}

/* =========================
   Markdown 전체
========================= */

.rendered_html{
    font-family:"마루 부리OTF 중간" !important;
    font-size:18px !important;
    line-height:1.8;
}

/* 제목 */
.rendered_html h1,
.rendered_html h2,
.rendered_html h3,
.rendered_html h4,
.rendered_html h5,
.rendered_html h6{
    font-family:"마루 부리OTF 조금굵은" !important;
}

/* 본문 */
.rendered_html p{
    font-family:"마루 부리OTF 중간" !important;
}

/* 리스트 */
.rendered_html li{
    font-family:"마루 부리OTF 중간" !important;
    padding:5px;
}

/* 인용 */
.rendered_html blockquote{
    font-family:"마루 부리OTF 중간" !important;
}

/* 표 */
.rendered_html table{
    font-family:"마루 부리OTF 중간" !important;
}

/* 코드 블록 */
.rendered_html pre,
.rendered_html code{
    font-family:"Consolas" !important;
    font-size:12pt !important;
}

</style>
"""))


# 1. tensorflow v2.xx에서 v1 사용하기

In [3]:
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior() # tensorflow v2 비활성화하고 v1만 활성화
import numpy as np
import pandas as pd

Instructions for updating:
non-resource variables are not supported in the long term


## Tensorflow
- 데이터 흐름 그래프(tensor 흐름을 나타내는 설계도)를 사용하는 수치 계산 라이브러리
- 그래프는 node(데이터, 연산)와 edge로 구성
- sess = tf.Session()을 이용하여 실행환경
- sess.run()을 통해서 실행결과를 확인

In [5]:
# 1. tensor(상수node, 변수node, 연산node) 정의
node1 = tf.constant('Hello, Tensorflow')

# 2. 세션 생성(연산을 실행하는 환경 생성)
sess = tf.Session()

# 3. 실행
print(sess.run(node1))
print(sess.run(node1).decode())

b'Hello, Tensorflow'
Hello, Tensorflow


In [6]:
# 간단한 연산 tensor 그래프

# 1. 그래프 정의
node1 = tf.constant(10, dtype=tf.float16)
node2 = tf.constant(20, dtype=tf.float16)
node3 = tf.add(node1, node2)

# 2. 세션 생성
sess = tf.Session()

# 3. 세션 실행 및 결과
n1, n2, n3 = (sess.run( [node1, node2, node3] ))
print(n1, n2, n3)

10.0 20.0 30.0


In [7]:
# 타입 변경
node1 = tf.constant(np.array( [10,20,30] ), dtype=tf.int16)
node2 = tf.cast(node1, dtype=tf.float32)
sess = tf.Session()
print(sess.run( [node1, node2] ))

[array([10, 20, 30], dtype=int16), array([10., 20., 30.], dtype=float32)]


In [13]:
# 평균값 계산 : tf.reduce_mean()
data = np.array( [1., 2, 3, 4] )
m = tf.reduce_mean(data)
sess = tf.Session()
sess.run(m)

2.5

In [17]:
# tf.random_noraml( [shape] ) : 평균0, 표준편차는 1인 shaoe 난수 배열 tensor. 기본값은 float32
w = tf.random.normal( [1] )
sess = tf.Session()
sess.run(w)

array([0.12529224], dtype=float32)

In [21]:
# 변수노드
w = tf.Variable( tf.random.normal([1]) )
sess = tf.Session()
sess.run(tf.global_variables_initializer()) # 난수가 발생될 변수 초기화
sess.run(w)

array([-2.6809216], dtype=float32)

# 2. tensorflow v1을 이용한 회귀분석 구현
## 2.1 독립(입력)변수 x가 1개, 종속(타겟)변수 y가 1개

In [37]:
# tensor 그래프 정의

# 데이터 셋 확보
x = np.array( [1,2,3] )
y = np.array( [2,3,4] )

# weight와 bias
w = tf.Variable( tf.random.normal([1]), name='weight' )
b = tf.Variable( tf.random.normal([1]), name='bias' )
# hat, hypothesis : 결과는 numpy배열
H = w * x + b
# cost function (손실함 : mse) : H-y의 제곱의 평균
cost = tf.reduce_mean( tf.square(H-y) )
'''
학습 목적: cost가 최소가 되는 w와 b를 찾는 것
cost함수가 2차 함수이므로 곡선 그래프, 곡선 위 미분값이 0이 되는 방향 학습(경사하강법:GradientDescent)
'''
optimizer = tf.train.GradientDescentOptimizer(learning_rate=0.01)
train = optimizer.minimize(cost)

# 세션 생성
sess = tf.Session()

# w와 b 초기화
sess.run(tf.global_variables_initializer())

# 학습(v2에서의 fit 함수)
for step in range(1, 7001):
    _, cost_val, w_val, b_val = sess.run( [train, cost, w, b] )
    if step%500==1:
        print(f'{step}번째 cost"{cost_val}, w:{w_val}, b:{b_val}')

1번째 cost"7.217811107635498, w:[2.628086], b:[-0.25157404]
501번째 cost"0.03530478850007057, w:[1.2177045], b:[0.50510633]
1001번째 cost"0.003180983243510127, w:[1.0653479], b:[0.8514489]
1501번째 cost"0.0002866123686544597, w:[1.0196155], b:[0.9554094]
2001번째 cost"2.582270280981902e-05, w:[1.0058877], b:[0.9866158]
2501번째 cost"2.327280753888772e-06, w:[1.0017676], b:[0.99598193]
3001번째 cost"2.1034269082065293e-07, w:[1.0005314], b:[0.9987922]
3501번째 cost"1.914981240247471e-08, w:[1.0001607], b:[0.9996356]
4001번째 cost"1.8177767069360584e-09, w:[1.0000495], b:[0.9998877]
4501번째 cost"1.787299197530956e-10, w:[1.0000156], b:[0.9999649]
5001번째 cost"5.328478283606053e-11, w:[1.0000087], b:[0.9999811]
5501번째 cost"5.328478283606053e-11, w:[1.0000087], b:[0.9999811]
6001번째 cost"5.328478283606053e-11, w:[1.0000087], b:[0.9999811]
6501번째 cost"5.328478283606053e-11, w:[1.0000087], b:[0.9999811]


In [43]:
w_, b_ = sess.run( [w[0], b[0]] )
w_, b_

(1.0000087, 0.9999811)

In [44]:
def predict(x):
    return x*w_ + b_

In [45]:
predict(5)

6.000024616718292

## 2.2 predict을 위한 placeholder이용
- placeholder : 외부에서 데이터를 입력받을 수 있는 노드

In [46]:
x = tf.placeholder(dtype=tf.float32)
H = w_*x + b_
sess = tf.Session()
sess.run([H, x], {x:2.5})

[3.5000029, array(2.5, dtype=float32)]

In [47]:
sess.run( H, {x: np.array([2, 3, 3]) } )

array([3.5000029, 4.000007 , 4.5000114], dtype=float32)

In [48]:
# tensor 그래프 정의

# 데이터 셋 확보
x_data = np.array( [1,2,3] )
y_data = np.array( [2,3,4] )

# placeholder 노드 설정
x = tf.placeholder(dtype=tf.float32)
y = tf.placeholder(dtype=tf.float32)

# weight와 bias
w = tf.Variable( tf.random.normal([1]), name='weight' )
b = tf.Variable( tf.random.normal([1]), name='bias' )

# hat, hypothesis : 결과는 numpy배열
H = w * x + b

# cost function (손실함 : mse) : H-y의 제곱의 평균
cost = tf.reduce_mean( tf.square(H-y) )
'''
학습 목적: cost가 최소가 되는 w와 b를 찾는 것
cost함수가 2차 함수이므로 곡선 그래프, 곡선 위 미분값이 0이 되는 방향 학습(경사하강법:GradientDescent)
'''
optimizer = tf.train.GradientDescentOptimizer(learning_rate=0.01)
train = optimizer.minimize(cost)

# 세션 생성
sess = tf.Session()

# w와 b 초기화
sess.run(tf.global_variables_initializer())

# 학습(v2에서의 fit 함수)
for step in range(1, 7001):
    _, cost_val, w_val, b_val = sess.run( [train, cost, w, b], feed_dict={x:x_data, y:y_data} )
    
    if step%500==1:
        print(f'{step}번째 cost"{cost_val}, w:{w_val}, b:{b_val}')

1번째 cost"18.33418083190918, w:[-0.2148047], b:[-0.24719374]
501번째 cost"0.004632533993571997, w:[1.0788605], b:[0.8207313]
1001번째 cost"0.0004173989873379469, w:[1.0236714], b:[0.9461892]
1501번째 cost"3.760552863241173e-05, w:[1.0071052], b:[0.9838483]
2001번째 cost"3.388958020877908e-06, w:[1.002133], b:[0.99515134]
2501번째 cost"3.058521826915239e-07, w:[1.0006411], b:[0.9985432]
3001번째 cost"2.786070218974146e-08, w:[1.0001936], b:[0.9995605]
3501번째 cost"2.5897293198795523e-09, w:[1.0000591], b:[0.99986625]
4001번째 cost"2.319069380973815e-10, w:[1.0000178], b:[0.99996006]
4501번째 cost"5.195962063386794e-11, w:[1.0000086], b:[0.9999813]
5001번째 cost"5.195962063386794e-11, w:[1.0000086], b:[0.9999813]
5501번째 cost"5.195962063386794e-11, w:[1.0000086], b:[0.9999813]
6001번째 cost"5.195962063386794e-11, w:[1.0000086], b:[0.9999813]
6501번째 cost"5.195962063386794e-11, w:[1.0000086], b:[0.9999813]


In [49]:
# 예측하기
sess.run(H, feed_dict={x:2.5})

array([3.5000029], dtype=float32)

In [50]:
sess.run( H, feed_dict={ x:np.array([2.5, 3.5]) } )

array([3.5000029, 4.5000114], dtype=float32)

## 2.3 scale이 다른 데이터들의 회귀분석 구현(scale조정X)

In [56]:
# tensor 그래프 정의

# 데이터 셋 확보
x_data = np.array( [1,2,5,8,10] )
y_data = np.array( [5,15,48,90,95] )

# placeholder 노드 설정
x = tf.placeholder(dtype=tf.float32)
y = tf.placeholder(dtype=tf.float32)

# weight와 bias
w = tf.Variable( tf.random.normal([1]), name='weight' )
b = tf.Variable( tf.random.normal([1]), name='bias' )

# hat, hypothesis : 결과는 numpy배열
H = w * x + b

# cost function (손실함 : mse) : H-y의 제곱의 평균
cost = tf.reduce_mean( tf.square(H-y) )
'''
학습 목적: cost가 최소가 되는 w와 b를 찾는 것
cost함수가 2차 함수이므로 곡선 그래프, 곡선 위 미분값이 0이 되는 방향 학습(경사하강법:GradientDescent)
'''
# optimizer = tf.train.GradientDescentOptimizer(learning_rate=0.01)
# train = optimizer.minimize(cost)
train = tf.train.GradientDescentOptimizer(learning_rate=0.01).minimize(cost)

# 세션 생성
sess = tf.Session()

# w와 b 초기화
sess.run(tf.global_variables_initializer())

# 학습(v2에서의 fit 함수)
for step in range(1, 6001):
    _, cost_val, w_val, b_val = sess.run( [train, cost, w, b], feed_dict={x:x_data, y:y_data} )
    
    if step%500==1:
        print(f'{step}번째 cost"{cost_val}')
print(f'{step}번째 cost:{cost_val}')

1번째 cost"3694.055419921875
501번째 cost"nan
1001번째 cost"nan
1501번째 cost"nan
2001번째 cost"nan
2501번째 cost"nan
3001번째 cost"nan
3501번째 cost"nan
4001번째 cost"nan
4501번째 cost"nan
5001번째 cost"nan
5501번째 cost"nan
6000번째 cost:nan


## 2.4 scale이 다른 데이터의 회귀분석(scale조정 O)
### scale조정방법 : 모든 데이터를 일정범위내로 조정
- normalization(정규화) : 모든 데이터를 0~1 사이로 조정
                                      X - Xmin
    normalization = ----------------------------------------
                                    Xmax - Xmin
      * 위의 식보다 라이브러리 추천(sklearn.preprocessing.StandardScaler)

In [64]:
# 라이브러리를 쓰지 않고 정규화
x_data = np.array( [1,2,5,8,10])
y_data = np.array( [5,15,48,90,95] )
norm_scaled_x_data = (x_data - x_data.min()) / (x_data.max() - x_data.min())
norm_scaled_y_data = (y_data - y_data.min()) / (y_data.max() - y_data.min())
print(norm_scaled_x_data)
print(norm_scaled_y_data)

[0.         0.11111111 0.44444444 0.77777778 1.        ]
[0.         0.11111111 0.47777778 0.94444444 1.        ]


In [62]:
# 라이브러리를 사용하여 정규화
from sklearn.preprocessing import MinMaxScaler, StandardScaler
x_data = np.array( [1,2,5,8,10] ).reshape(-1, 1)
y_data = np.array( [5,15,48,90,95] ).reshape(-1, 1)
scaler_x = MinMaxScaler() # x_data를 변환시킬 객체
scaler_x.fit(x_data)
norm_scaled_x_data = scaler_x.transform(x_data)
scaler_y = MinMaxScaler() # y_data를 변환시킬 객체
# scaler_y.fit(y_data)
# norm_scaled_y_data = scaler_y.transform(y_data)
norm_scaled_y_data = scaler_y.fit_transform(y_data)
np.column_stack( [x_data, norm_scaled_x_data, y_data, norm_scaled_y_data])

array([[ 1.        ,  0.        ,  5.        ,  0.        ],
       [ 2.        ,  0.11111111, 15.        ,  0.11111111],
       [ 5.        ,  0.44444444, 48.        ,  0.47777778],
       [ 8.        ,  0.77777778, 90.        ,  0.94444444],
       [10.        ,  1.        , 95.        ,  1.        ]])

In [63]:
# 라이브러리를 쓰지 않고 표준화
x_data = np.array( [1,2,5,8,10])
y_data = np.array( [5,15,48,90,95] )
stan_scaled_x_data = ( x_data - x_data.mean() ) / x_data.std()
stan_scaled_y_data = ( y_data - y_data.mean() ) / y_data.std()
print(np.column_stack( [x_data, stan_scaled_x_data, norm_scaled_x_data] ))
print()
print(np.column_stack( [y_data, stan_scaled_y_data, norm_scaled_y_data] ))

[[ 1.         -1.22474487  0.        ]
 [ 2.         -0.93313895  0.11111111]
 [ 5.         -0.05832118  0.44444444]
 [ 8.          0.81649658  0.77777778]
 [10.          1.39970842  1.        ]]


In [65]:
# 라이브러리를 사용하여 표준화
x_data = np.array( [1,2,5,8,10] ).reshape(-1, 1)
y_data = np.array( [5,15,48,90,95] ).reshape(-1, 1)
scaler_x = StandardScaler()
stan_scaled_x_data = scaler_x.fit_transform(x_data)
scaler_y = StandardScaler()
stan_scaled_y_data = scaler_y.fit_transform(y_data)
np.column_stack( [stan_scaled_x_data, stan_scaled_y_data] )

array([[-1.22474487, -1.22954384],
       [-0.93313895, -0.95990703],
       [-0.05832118, -0.07010557],
       [ 0.81649658,  1.06236902],
       [ 1.39970842,  1.19718742]])

In [66]:
# 스케일 조정된 데이터를 다시 복구 : inverse_transform() 이용
scaler_x.inverse_transform(stan_scaled_x_data)

array([[ 1.],
       [ 2.],
       [ 5.],
       [ 8.],
       [10.]])

In [68]:
scaler_y.inverse_transform(stan_scaled_y_data)

array([[ 5.],
       [15.],
       [48.],
       [90.],
       [95.]])

In [71]:
# tensor 그래프 정의

# 데이터 셋 확보
x_data = np.array( [1,2,5,8,10] )
y_data = np.array( [5,15,48,90,95] )

# placeholder 노드 설정
x = tf.placeholder(dtype=tf.float32)
y = tf.placeholder(dtype=tf.float32)

# weight와 bias
w = tf.Variable( tf.random.normal([1]), name='weight' )
b = tf.Variable( tf.random.normal([1]), name='bias' )

# hat, hypothesis : 결과는 numpy배열
H = w * x + b

# cost function (손실함 : mse) : H-y의 제곱의 평균
cost = tf.reduce_mean( tf.square(H-y) )
'''
학습 목적: cost가 최소가 되는 w와 b를 찾는 것
cost함수가 2차 함수이므로 곡선 그래프, 곡선 위 미분값이 0이 되는 방향 학습(경사하강법:GradientDescent)
'''
# optimizer = tf.train.GradientDescentOptimizer(learning_rate=0.01)
# train = optimizer.minimize(cost)
train = tf.train.GradientDescentOptimizer(learning_rate=0.01).minimize(cost)

# 세션 생성
sess = tf.Session()

# w와 b 초기화
sess.run(tf.global_variables_initializer())

# 학습(v2에서의 fit 함수)
for step in range(1, 2001):
    _, cost_val, w_val, b_val = sess.run( 
        [train, cost, w, b],
            feed_dict={
                x:stan_scaled_x_data, 
                y:stan_scaled_y_data} 
    )
    
    if step%400==1:
        print(f'{step}번째 cost"{cost_val}')
print(f'{step}번째 cost:{cost_val}')

1번째 cost"0.941157341003418
401번째 cost"0.020364627242088318
801번째 cost"0.02036452293395996
1201번째 cost"0.02036452852189541
1601번째 cost"0.02036452852189541
2000번째 cost:0.02036452852189541


## 2.5 독립변수 x가 3개, 타겟변수 y가 1개 회귀분석